# E09 03 - RAG + agentes con LangGraph + Langfuse (Starter)

Este ejercicio introduce RAG y agentes de forma minima.

RAG significa:

```txt
pregunta -> retriever -> contexto -> prompt -> LLM -> respuesta fundamentada
```

La idea central: el modelo no responde solo con memoria interna; primero recibe documentos recuperados.


## Antes de tocar codigo: que estamos construyendo

Este notebook esta pensado para que puedas entenderlo aunque lo abras sin ver la clase.

Tema: **RAG + agentes con LangGraph + Langfuse**.

La regla didactica es:

1. primero explicamos el concepto;
2. despues mostramos el codigo minimo;
3. despues conectamos ese codigo con el paso anterior;
4. finalmente ejecutamos y leemos el resultado.

Cuando veas una funcion, preguntate:

- que recibe;
- que devuelve;
- que parte del flujo representa;
- si es logica de negocio, orquestacion o instrumentacion.


## Herramientas RAG que aparecen

| Herramienta | Para que sirve |
|---|---|
| `Document` | Guarda texto y metadata |
| `RecursiveCharacterTextSplitter` | Divide documentos en chunks |
| `OpenAIEmbeddings` | Convierte texto en vectores |
| `Chroma` | Vector store para buscar por similitud |
| `retriever` | Interfaz de busqueda semantica |
| `ChatPromptTemplate` | Prompt que combina contexto + consulta |


In [ ]:
# Esta celda instala las dependencias del notebook.
# En Google Colab cada notebook arranca con un entorno limpio, por eso instalamos al inicio.
# En local, si ya instalaste estos paquetes, pip simplemente confirmara que existen.
!pip install -q langchain langchain-openai langchain-chroma chromadb langgraph langfuse

print('Dependencias instaladas: langchain langchain-openai langchain-chroma chromadb langgraph langfuse')


In [ ]:
import os
from getpass import getpass

# Langfuse usa dos credenciales del proyecto: public key y secret key.
# Se obtienen en Langfuse Cloud, dentro de Settings -> API Keys.
os.environ['LANGFUSE_PUBLIC_KEY'] = getpass('Langfuse Public Key: ')
os.environ['LANGFUSE_SECRET_KEY'] = getpass('Langfuse Secret Key: ')

# URL del servicio. Para la region US se puede cambiar por https://us.cloud.langfuse.com.
os.environ['LANGFUSE_BASE_URL'] = 'https://cloud.langfuse.com'

# OpenAI sigue siendo necesario porque el grafo llama al LLM.
os.environ['OPENAI_API_KEY'] = getpass('OpenAI API Key: ')

print('OpenAI y Langfuse configurados para esta sesion.')


## Imports

Importamos piezas de RAG, LLM y, si corresponde, LangGraph/Langfuse.


In [ ]:
from typing import TypedDict
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langgraph.graph import StateGraph, START, END
from langfuse.langchain import CallbackHandler

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')


## Documentos

En un proyecto real vendrian de archivos. Aca los dejamos en el notebook para que sea autocontenido.


In [ ]:
from langchain_core.documents import Document

# Document es el contenedor estandar de LangChain para texto + metadata.
# page_content guarda el texto que se va a recuperar.
# metadata guarda informacion util como dominio y fuente.
docs = [
    Document(page_content='RRHH: las vacaciones se piden con 15 dias de anticipacion y aprobacion del manager.', metadata={'domain': 'hr', 'source': 'hr_policy'}),
    Document(page_content='RRHH: la licencia por estudio requiere constancia de examen y registro en PeopleOps.', metadata={'domain': 'hr', 'source': 'hr_policy'}),
    Document(page_content='Tech: para problemas de VPN, reiniciar cliente, validar 2FA y probar otra red.', metadata={'domain': 'tech', 'source': 'tech_guide'}),
    Document(page_content='Tech: el reset de contrasena se realiza desde el portal de identidad corporativa.', metadata={'domain': 'tech', 'source': 'tech_guide'}),
    Document(page_content='Finance: los reembolsos aprobados se procesan los dias 10 y 25 de cada mes.', metadata={'domain': 'finance', 'source': 'finance_policy'}),
    Document(page_content='Finance: las facturas deben cargarse con comprobante, monto y centro de costo.', metadata={'domain': 'finance', 'source': 'finance_policy'}),
]

print('Documentos cargados:', len(docs))


## Chunks y ChromaDB

El splitter crea fragmentos. ChromaDB guarda embeddings y permite recuperar los fragmentos mas parecidos a la pregunta.


In [ ]:
# TODO 1: crear RecursiveCharacterTextSplitter.
splitter = None

# TODO 2: partir docs en chunks.
chunks = None

# TODO 3: crear Chroma.from_documents con chunks y embeddings.
vectorstore = None

# TODO 4: crear retriever con vectorstore.as_retriever(search_kwargs={'k': 2}).
retriever = None


## Prompt RAG

El prompt obliga al LLM a usar solo el contexto recuperado.


In [ ]:
# TODO 5: crear ChatPromptTemplate con variables {context} y {query}.
rag_prompt = None

# TODO 6: crear una funcion format_docs(docs) que una page_content.
def format_docs(retrieved_docs):
    raise NotImplementedError('Completar format_docs')


## Estado y agentes

Ahora el RAG vive dentro de nodos especialistas.


In [ ]:
class RAGState(TypedDict):
    query: str
    intent: str
    context: str
    answer: str

def route_query(query: str) -> str:
    # TODO 7: router simple por keywords: hr, tech, finance o unknown.
    raise NotImplementedError('Completar route_query')

def router_node(state: RAGState) -> dict:
    # TODO 8: devolver {'intent': route_query(state['query'])}.
    raise NotImplementedError('Completar router_node')

def rag_node(domain: str, state: RAGState) -> dict:
    # TODO 9: recuperar, formatear y responder con rag_chain.
    raise NotImplementedError('Completar rag_node')


In [ ]:
# TODO 10: crear hr_node, tech_node, finance_node y unknown_node.
# TODO 11: crear route_to_node.
# TODO 12: construir StateGraph con add_conditional_edges.
graph = None

# TODO 13: crear CallbackHandler e invocar con callbacks.
langfuse_handler = None
